# Camada Gold: Business Intelligence e Data Marts

O objetivo deste notebook é consolidar as informações das camadas anteriores em tabelas analíticas finais (Data Marts), prontas para consumo por ferramentas de visualização e tomada de decisão. Nesta etapa, aplicamos as últimas agregações para responder a perguntas estratégicas de negócio.

**Principais Entregas:**
* **Visão Comercial:** Agrupamento de vendas por granularidade temporal (Ano/Mês) e categoria de produto.
* **Métricas Financeiras:** Cálculo de Receita Total em BRL/USD e Ticket Médio por transação.
* **Data Mart de Satisfação:** Consolidação de notas de avaliações por vendedor, cidade e estado.
* **Rankings Estratégicos:** Identificação dos melhores e piores produtos e vendedores com base em performance e qualidade.

In [0]:
# Criar banco de dados Gold
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

from pyspark.sql.functions import col, count, countDistinct, sum, avg, round, month, year, expr

### Análise de Performance Comercial
Este bloco consolida os dados de vendas, itens e produtos para gerar uma visão histórica do faturamento. A granularidade é definida por Ano, Mês e Categoria de Produto, permitindo identificar tendências sazonais e o desempenho de cada nicho de mercado. Os valores são arredondados para duas casas decimais para garantir precisão nos relatórios financeiros.

In [0]:
from pyspark.sql.functions import round as spark_round, count, col

# Lendo as tabelas da Silver
df_vendas = spark.read.table("silver.fat_pedido_total")
df_itens = spark.read.table("silver.fat_itens_pedidos")
df_produtos = spark.read.table("silver.dim_produtos")

# Juntar para ter as categorias e métricas (Granularidade: Ano, Mês, Categoria)
df_comercial = df_vendas.join(df_itens, "id_pedido") \
    .join(df_produtos, "id_produto")

gold_vendas = df_comercial.groupBy(
    year("data_pedido").alias("ano_venda"),
    month("data_pedido").alias("mes_venda"),
    "categoria_produto"
).agg(
    countDistinct("id_pedido").alias("total_pedidos"),
    count("id_item").alias("qtd_itens_vendidos"),
    spark_round(sum("valor_total_pago_brl"), 2).alias("receita_total_brl"),
    spark_round(sum("valor_total_pago_usd"), 2).alias("receita_total_usd")
).withColumn("ticket_medio_brl", spark_round(col("receita_total_brl") / col("total_pedidos"), 2))

# Salvar na Gold (Modo Overwrite é obrigatório)
gold_vendas.write.format("delta").mode("overwrite").saveAsTable("gold.fat_vendas_comercial")

print("Tabela gold.fat_vendas_comercial criada!")

# Entrega 2 - Rankings Comerciais (Refatorado para remover a coluna 'count')
print("Top 5 Produtos MAIS Vendidos:")
display(df_comercial.groupBy("id_produto", "categoria_produto") \
    .agg(count("id_item").alias("quantidade_vendas")) \
    .orderBy(col("quantidade_vendas").desc()).limit(5))

print("Top 5 Produtos MENOS Vendidos:")
display(df_comercial.groupBy("id_produto", "categoria_produto") \
    .agg(count("id_item").alias("quantidade_vendas")) \
    .orderBy(col("quantidade_vendas").asc()).limit(5))

Tabela gold.fat_vendas_comercial criada!
Top 5 Produtos MAIS Vendidos:


id_produto,categoria_produto,quantidade_vendas
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,ferramentas_jardim,388


Top 5 Produtos MENOS Vendidos:


id_produto,categoria_produto,quantidade_vendas
e9f2586707fb45ea0f9997c54f585842,esporte_lazer,1
e1c0a39b7f806727ea5121c60035ed3c,informatica_acessorios,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
8d7ab3701456fdbfe2526636ce15327a,malas_acessorios,1
cdb8d3c880b6639a70764c6734e6bb69,beleza_saude,1


### Análise de Qualidade e Satisfação do Cliente
Neste estágio, construímos uma "ponte" de dados que conecta as avaliações dos clientes diretamente aos vendedores e produtos correspondentes. O objetivo é calcular a média de satisfação, o volume de feedbacks e a taxa de avaliações positivas vs. negativas. Estes KPIs são fundamentais para a gestão de qualidade da plataforma e identificação de pontos de melhoria logística ou de atendimento.

In [0]:
from pyspark.sql.functions import round as spark_round, avg, count, when, col

# 1. Lendo as tabelas da Silver (Certifique-se de que dim_produtos está aqui)
df_reviews = spark.read.table("silver.fat_avaliacoes_pedidos")
df_itens = spark.read.table("silver.fat_itens_pedidos")
df_vendedores = spark.read.table("silver.dim_vendedores")
df_vendas_final = spark.read.table("silver.fat_pedido_total")
df_produtos = spark.read.table("silver.dim_produtos") # Adicionado aqui

# 2. Criando a "Ponte" COMPLETA: Avaliação -> Pedido -> Item -> Vendedor -> PRODUTO
df_qualidade_base = df_reviews.join(df_itens, "id_pedido") \
    .join(df_vendedores, "id_vendedor") \
    .join(df_vendas_final, "id_pedido") \
    .join(df_produtos, "id_produto") # Adicionado aqui para trazer a categoria

# 3. Agrupando para o Data Mart de Satisfação (Vendedores)
gold_satisfacao = df_qualidade_base.groupBy("id_vendedor", "cidade", "estado").agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    spark_round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    count(when(col("nota_avaliacao") >= 4, 1)).alias("total_avaliacoes_positivas"),
    count(when(col("nota_avaliacao") <= 2, 1)).alias("total_avaliacoes_negativas")
).withColumn("percentual_satisfacao", spark_round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2))

# 4. Salvar na Gold conforme o PDF [cite: 145]
gold_satisfacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_avaliacoes_clientes")

# --- Ranking de Qualidade por PRODUTO (Agora com categoria_produto resolvida) ---
gold_satisfacao_produto = df_qualidade_base.groupBy("id_produto", "categoria_produto").agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    spark_round(avg("nota_avaliacao"), 2).alias("avaliacao_media")
)

print("1. O Produto MAIS bem avaliado:")
display(gold_satisfacao_produto.orderBy(col("avaliacao_media").desc(), col("total_avaliacoes").desc()).limit(1))

print("2. O Produto MENOS bem avaliado:")
display(gold_satisfacao_produto.orderBy(col("avaliacao_media").asc(), col("total_avaliacoes").desc()).limit(1))

print("3. O Vendedor MAIS bem avaliado:")
display(gold_satisfacao.orderBy(col("avaliacao_media").desc(), col("total_avaliacoes").desc()).limit(1))

print("4. O Vendedor MENOS bem avaliado:")
display(gold_satisfacao.orderBy(col("avaliacao_media").asc(), col("total_avaliacoes").desc()).limit(1))

1. O Produto MAIS bem avaliado:


id_produto,categoria_produto,total_avaliacoes,avaliacao_media
37eb69aca8718e843d897aa7b82f462d,ferramentas_jardim,15,5.0


2. O Produto MENOS bem avaliado:


id_produto,categoria_produto,total_avaliacoes,avaliacao_media
05b515fdc76e888aada3c6d66c201dff,beleza_saude,10,1.0


3. O Vendedor MAIS bem avaliado:


id_vendedor,cidade,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
48efc9d94a9834137efd9ea76b065a38,CURITIBA,PR,32,5.0,32,0,100.0


4. O Vendedor MENOS bem avaliado:


id_vendedor,cidade,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
8d92f3ea807b89465643c219455e7369,SAO PAULO,SP,8,1.0,0,8,0.0
